In [1]:
import os
from pathlib import Path
import pandas as pd

# 1. Flexible path configuration
BASE_DIR = Path.cwd().parent
RAW_DATA_PATH = BASE_DIR / "data" / "raw" / "results.csv"

# In some Jupyter setups, cwd is already the project root
if not RAW_DATA_PATH.exists():
    BASE_DIR = Path.cwd()
    RAW_DATA_PATH = BASE_DIR / "data" / "raw" / "results.csv"

# Output paths for our split datasets
PROCESSED_HISTORICAL_PATH = BASE_DIR / "data" / "processed" / "historical_matches.csv"
PROCESSED_FIXTURES_PATH = BASE_DIR / "data" / "processed" / "upcoming_fixtures.csv"

def run_data_ingestion_pipeline():
    if not RAW_DATA_PATH.exists():
        print(f"[ERROR] Source file not found: {RAW_DATA_PATH}")
        return

    print("[INFO] Loading raw dataset...")
    df = pd.read_csv(RAW_DATA_PATH)
    initial_count = len(df)

    # 2. Split the data based on missing scores (The 'Aha!' moment)
    # Rows with valid scores go to history, rows with NA scores go to fixtures
    df_historical = df.dropna(subset=["home_score", "away_score"]).copy()
    df_fixtures = df[df["home_score"].isna() | df["away_score"].isna()].copy()

    # 3. Clean Historical Data (For Elo Engine)
    df_historical = df_historical.dropna(subset=["home_team", "away_team", "tournament"])
    df_historical = df_historical.astype({"home_score": "int32", "away_score": "int32"})
    df_historical["date"] = pd.to_datetime(df_historical["date"])
    
    # Chronological sort is CRITICAL for calculating rolling Elo ratings
    df_historical = df_historical.sort_values(by="date").reset_index(drop=True)

    # 4. Clean Fixtures Data (For Bracket Simulator Predictions)
    # We filter only the 2026 World Cup future matches
    df_fixtures = df_fixtures[df_fixtures["tournament"] == "FIFA World Cup"]
    df_fixtures["date"] = pd.to_datetime(df_fixtures["date"])
    df_fixtures = df_fixtures.sort_values(by="date").reset_index(drop=True)
    
    # Drop the unnecessary score columns for future games
    df_fixtures = df_fixtures.drop(columns=["home_score", "away_score"])

    # 5. Save processed files
    PROCESSED_HISTORICAL_PATH.parent.mkdir(parents=True, exist_ok=True)
    df_historical.to_csv(PROCESSED_HISTORICAL_PATH, index=False)
    df_fixtures.to_csv(PROCESSED_FIXTURES_PATH, index=False)

    # 6. Engineering Report
    print(" DATA PIPELINE EXECUTION SUMMARY")
    print("="*50)
    print(f" Total raw records parsed: {initial_count}")
    print(f" Valid historical matches: {len(df_historical)}")
    print(f" Upcoming WC fixtures:     {len(df_fixtures)}")
    print("="*50)
    print("Pipeline completed. Data ready for Elo Engine and Predictor.")

# Execute the pipeline
run_data_ingestion_pipeline()

[INFO] Loading raw dataset...
 DATA PIPELINE EXECUTION SUMMARY
 Total raw records parsed: 49494
 Valid historical matches: 49484
 Upcoming WC fixtures:     10
Pipeline completed. Data ready for Elo Engine and Predictor.
